# Setup

In [17]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/spark"

# configuration
LOGGING = True # enable logging, slower but more detailed on data amount and format
SEED = 25 # seed for reproducibility (None for random seed)
SAMPLE_SIZE = 1.0 # fraction of the dataset used (between 0.0 and 1.0)
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_SKETCHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert 0 < SAMPLE_SIZE <= 1, "invalid SAMPLE_SIZE, must be between 0.0 and 1.0"
assert LSH_NUM_SKETCHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_SKETCHES, must be equal to LSH_BANDS * LSH_ROWS"


In [18]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [19]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
      .builder
      .getOrCreate())
sc = ss.sparkContext


Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [20]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped as ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = (ss
                 .read
                 .options(**csv_options)
                 .csv("dataset/nyt-articles-2020.csv")
                 .sample(withReplacement=False, fraction=SAMPLE_SIZE, seed=SEED))
full_comments = (ss
                 .read
                 .options(**csv_options)
                 # .csv("dataset/nyt-comments-2020.csv")
                 .csv("dataset/nyt-comments-part0.csv") # subset of comments
                 .sample(withReplacement=False, fraction=SAMPLE_SIZE, seed=SEED))

if LOGGING:
    print(f"Number of articles: {full_articles.count()}")
    print(f"Number of comments: {full_comments.count()}")
    print(f"Sample article: {full_articles.take(1)[0]}")
    print(f"Sample comment: {full_comments.take(1)[0]}")


Number of articles: 16787
Number of comments: 500000
Sample article: Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')
Sample comment: Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan opti

In [21]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data, seed=0 if SEED is None else SEED)


Note: you may need to restart the kernel to use updated packages.


## RDD

In [22]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))
users = (comments
         .map(lambda x: x[1])
         .distinct())

if LOGGING:
    print(f"Number of articles: {articles.count()}")
    print(f"Number of comments: {comments.count()}")
    print(f"Number of users: {users.count()}")
    print(f"Sample article: {articles.take(1)[0]}")
    print(f"Sample comment: {comments.take(1)[0]}")


Number of articles: 16787
Number of comments: 500000
Number of users: 91437
Sample article: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 'Editorial', 'Opinion', None, ['veterans', 'for-profit schools', 'financial aid (education)', 'frauds and swindling', 'colleges and universities', 'veterans affairs department', 'federal trade commission', 'university of phoenix', 'career education corporation'])
Sample comment: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)


# Building Profiles

## Utility Matrix

In [23]:
utility_matrix = (comments
                  .distinct()) # ignore multiple comments from same user on same article

if LOGGING:
    print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
    print(f"Sample entry in utility matrix: {utility_matrix.take(1)[0]}")


Number of entries in sparse utility matrix: 356283
Sample entry in utility matrix: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)


## Articles Profile - via dictionary

In [24]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = pruned_keywords + (articles
                              .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
                              .distinct()
                              .collect())

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
broadcast_dict = sc.broadcast(features_mapping)

if LOGGING:
    print(f"Extacted {len(features_mapping)} features (broadcasting ~{((100 * 4) + 4) * len(features_mapping) // 1024} KB)")

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = (articles
                         .flatMap(build_article_profile_dict) # (article_id, feature_index)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_index, frequency = 1))

if LOGGING:
    print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
    print(f"Sample entry in sparse articles profile via dictionary: {articles_profile_dict.take(1)[0]}")


Extacted 3555 features (broadcasting ~1402 KB)
Number of entries in sparse articles profile via dictionary: 158079
Sample entry in sparse articles profile via dictionary: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3384, 1))


## Articles Profile - via hashing

In [25]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = (articles
                         .flatMap(build_article_profile_hash) # (article_id, feature_hash)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_hash, frequency = 1))

if LOGGING:
    print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
    print(f"Sample entry in sparse articles profile via hash: {articles_profile_hash.take(1)[0]}")


Number of entries in sparse articles profile via hash: 185396
Sample entry in sparse articles profile via hash: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (592, 1))


## WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [26]:
# articles_profile = articles_profile_dict
articles_profile = articles_profile_hash


## Users Profile

In [27]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                 .join(articles_profile) # (article_id, (user_id, (feature_index, frequency = 1)))
                 .map(lambda x: ((x[1][0], x[1][1][0]), 1)) # ((user_id, feature_index), 1)
                 .reduceByKey(lambda a, b: a + b) # ((user_id, feature_index), frequency)
                 .map(lambda x: (x[0][0], (x[0][1], x[1])))) # (user_id, (feature_index, frequency))

if LOGGING:
    print(f"Number of entries in sparse users profile: {users_profile.count()}")
    print(f"Sample entry in sparse users profile: {users_profile.take(1)[0]}")


Number of entries in sparse users profile: 2818910
Sample entry in sparse users profile: (5646097, (4379, 36))


# Computing Similarity

## Profiles sketches (signatures)

In [28]:
import random
if SEED is not None: random.seed(SEED)

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_SKETCHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)

if LOGGING:
    print(f"Generated {LSH_NUM_SKETCHES} random hyperplanes (broadcasting ~{(LSH_NUM_SKETCHES * FEATURES_SIZE * 4) // 1024} KB)")

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile # (user_id, (feature_index, frequency))
                  .groupByKey()
                  .mapValues(list) # (user_id, [(feature_index, frequency), ...])
                  .map(compute_sketch)) # (user_id, sketch)
articles_sketches = (articles_profile # (article_id, (feature_index, frequency = 1))
                     .groupByKey()
                     .mapValues(list) # (article_id, [(feature_index, frequency), ...])
                     .map(compute_sketch)) # (article_id, sketch)

if LOGGING:
    print(f"Number of users sketches: {users_sketches.count()}")
    print(f"Number of articles sketches: {articles_sketches.count()}")
    print(f"Sample users sketch: {users_sketches.take(1)[0]}")
    print(f"Sample articles sketch: {articles_sketches.take(1)[0]}")


Generated 100 random hyperplanes (broadcasting ~1953 KB)
Number of users sketches: 91437
Number of articles sketches: 16787
Sample users sketch: (67789834, [0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0])
Sample articles sketch: ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', [1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1])


## LSH buckets

In [29]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id)) # no need to differentiate between users and articles: different format and distinct rdds
    return res

users_buckets = users_sketches.flatMap(lsh_buckets) # (bucket_hash, user_id)
articles_buckets = articles_sketches.flatMap(lsh_buckets) # (bucket_hash, article_id)

if LOGGING:
    print(f"Total number of users buckets: {users_buckets.count()}")
    print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets # (bucket_hash, user_id)
                 .join(valid_users_buckets) # (bucket_hash, (user_id, count))
                 .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, user_id)
articles_buckets = (articles_buckets # (bucket_hash, article_id)
                    .join(valid_articles_buckets) # (bucket_hash, (article_id, count))
                    .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, article_id)

if LOGGING:
    print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} users) number of users buckets: {users_buckets.count()}")
    print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} articles) number of articles buckets: {articles_buckets.count()}")
    print(f"Sample users sparse bucket: {users_buckets.take(1)[0]}")
    print(f"Sample articles sparse bucket: {articles_buckets.take(1)[0]}")


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid (less than 500 users) number of users buckets: 549068
Valid (less than 500 articles) number of articles buckets: 161078
Sample users sparse bucket: (2009018280, 228594)
Sample articles sparse bucket: (1991176582, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')


## LSH filter

In [30]:
lsh_recommendations = (users_buckets # (bucket_hash, user_id)
                       .join(articles_buckets) # (bucket_hash, (user_id, article_id))
                       .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
                       .distinct() # remove matches from multiple bands
                       .subtract(utility_matrix.map(lambda x: (x[1], x[0])))) # remove already commented articles

if LOGGING:
    print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
    print(f"Sample sparse recommendation that passed LSH filter: {lsh_recommendations.take(1)[0]}")


Number of recommendations that passed LSH filter: 12483493
Sample sparse recommendation that passed LSH filter: (67866393, 'nyt://article/c091f18f-b70a-5331-be9f-7e88e23b27df')


## Actual Cosine Similarity

In [31]:
# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0))

users_profile.unpersist()
articles_profile.unpersist()

if LOGGING:
    print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
    print(f"Sample recommendation with actual cosine similarity: {recommendations.take(1)[0]}")


Number of recommendations with actual cosine similarity: 6171001
Sample recommendation with actual cosine similarity: (36052407, ('nyt://article/99362592-e5d5-5d11-b5c5-d7964bef5be7', 0.13245323570650439))


# Content-Based Recommendations

In [32]:
import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1]))
                        .cache())

print(f"Number of users with at least one recommendation: {user_recommendations.count()}/{users.count()}")

if LOGGING:
    print(f"Sample user recommendations: {user_recommendations.take(1)[0]}")


Number of users with at least one recommendation: 83601/91437
Sample user recommendations: (7304670, [('nyt://article/a038f50a-a6b9-5c39-a128-ae4975c9f3f6', 0.33472), ('nyt://article/6a03958e-0edd-5c89-acf8-6e9bf3077209', 0.32159), ('nyt://article/f3e787bc-0a92-5174-a369-0a6e094e2eee', 0.29235), ('nyt://article/965e5363-25d9-5aed-9086-f80080525b32', 0.28604), ('nyt://article/1d487a21-8e74-589c-aaba-7dd62396c794', 0.28109), ('nyt://article/6bc0b816-6248-5131-a4f2-cee9fae576f3', 0.28109), ('nyt://article/5bab1d6a-2c57-5040-8a88-dc9935555b6f', 0.27386), ('nyt://article/3578854e-5d6f-5e14-b607-60c94da623c0', 0.27217), ('nyt://article/5e62ca8f-ae51-5c14-8b93-7a22635ef49c', 0.26667), ('nyt://article/25dbf43c-3a82-5f1d-9a2a-6169550d5e91', 0.26667)])


## Human Evaluation

In [33]:
USER = users.takeSample(False, 1, seed=SEED)[0]
# USER = 7779131 # one comment, good reccs
# USER = 22181262 # one comment, perfect matches
# USER = 60191297 # a lot of comments
# USER = 60624667 # two very different comments
# USER = users.takeSample(False, 1)[0] # random user

print(f"Selected user: {USER}")

read = (utility_matrix
        .filter(lambda x: x[1] == USER)
        .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, (user_id, article_info))
        .map(lambda x: x[1][1]) # (article_info)
        .collect())

reccs = (user_recommendations
         .filter(lambda x: x[0] == USER)
         .map(lambda x: x[1])
         .flatMap(lambda x: x)
         .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, article_info)
         .map(lambda x: (x[1][1], x[1][0])) # (article_info, similarity)
         .collect())

print("--- ARTICLES COMMENTED ---")
for r in read:
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")

print("--- ARTICLES RECOMMENDED ---")
for r, s in sorted(reccs, key=lambda x: x[1], reverse=True):
    kws = "\n\t".join(parse_keywords(r))
    print(f"  {round(s, 5)} {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{kws}")


Selected user: 59879418
--- ARTICLES COMMENTED ---
  Biden, Sanders, Social Security and Smears
	OpEd - Opinion - None
	presidential election of 2020
	united states politics and government
	social security (us)
	democratic party
	biden
	joseph r jr
	sanders
	bernard
	trump
	donald j
  If Bernie Wins, Where Will He Take the Democratic Party?
	OpEd - Opinion - None
	presidential election of 2020
	presidential election of 2016
	primaries and caucuses
	midterm elections (2018)
	united states politics and government
	democratic party
	buttigieg
	pete (1982- )
	chait
	jonathan
	biden
	joseph r jr
	french
	david a
	sanders
	bernard
	trump
	donald j
--- ARTICLES RECOMMENDED ---
  0.78567 Have We Learned Nothing After Four Years of Trump?
	OpEd - Opinion - None
	presidential election of 2020
	conservatism (us politics)
	democratic party
	republican party
	biden
	joseph r jr
	trump
	donald j
	sanders
	bernard
  0.6931 I Asked Bernie Sanders if It Was All Over. ‘No,’ He Groaned.
	OpEd - Opinion -